## Positional encoding 

In [5]:
import torch
import torch.nn as nn
import math

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        """
        d_model : embedding dimension (e.g., 512)
        max_len : maximum sequence length we expect
        """
        super(PositionalEncoding, self).__init__()

        # Create a matrix of shape (max_len, d_model)
        # This will store positional encodings for all positions
        pe = torch.zeros(max_len, d_model)

        # Create position indices: [0, 1, 2, ..., max_len-1]
        # Shape becomes (max_len, 1)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)

        # Compute the divisor term (frequency term)
        # This controls how fast sin/cos oscillate
        # Only applied to even indices (0,2,4,...)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )

        # Apply sin to even indices (0,2,4,...)
        pe[:, 0::2] = torch.sin(position * div_term)

        # Apply cos to odd indices (1,3,5,...)
        pe[:, 1::2] = torch.cos(position * div_term)

        # Add batch dimension
        # Final shape: (max_len, 1, d_model)
        pe = pe.unsqueeze(1)

        # Register as buffer (not trainable, but moves with model)
        self.register_buffer('pe', pe)

    def forward(self, x):
        """
        x shape: (seq_len, batch_size, d_model)

        We add positional encoding to embeddings.
        """
        # Add positional encoding to input embeddings
        x = x + self.pe[:x.size(0)]
        return x

In [4]:
# Example usage
d_model = 6
seq_len = 4
batch_size = 1

# Create dummy input embeddings (all zeros for clarity)
x = torch.zeros(seq_len, batch_size, d_model)

# Initialize positional encoding
pos_encoder = PositionalEncoding(d_model=d_model, max_len=10)

# Apply positional encoding
output = pos_encoder(x)

print("Output shape:", output.shape)
print("Output tensor:\n", output)

Output shape: torch.Size([4, 1, 6])
Output tensor:
 tensor([[[ 0.0000,  1.0000,  0.0000,  1.0000,  0.0000,  1.0000]],

        [[ 0.8415,  0.5403,  0.0464,  0.9989,  0.0022,  1.0000]],

        [[ 0.9093, -0.4161,  0.0927,  0.9957,  0.0043,  1.0000]],

        [[ 0.1411, -0.9900,  0.1388,  0.9903,  0.0065,  1.0000]]])
